In [ ]:
#This code makes a moving average of the time series

import pandas as pd
import numpy as np

ids=range(29, 56, 1)

window_size = 31
threshold_std = 2

def validate_and_smooth(series, window=5, threshold=2):
    smoothed = series.copy()
    half_window = window // 2

    for i in range(len(series)):
        start = max(0, i - half_window)
        end = min(len(series), i + half_window + 1)

        window_vals = series[start:end].copy()
        center_val = series.iloc[i]

        surrounding_vals = window_vals.drop(series.index[i])
        surrounding_vals = surrounding_vals.dropna()

        if np.isnan(center_val) or len(surrounding_vals) < 5:
            continue

        mean = surrounding_vals.mean()
        std = surrounding_vals.std()

        if abs(center_val - mean) > threshold * std:
            smoothed.iloc[i] = None
        
        else:
            smoothed.iloc[i] = mean
    return smoothed

for id in ids:
    df = pd.read_csv(f"csv_by_baby_filtered/baby_{id}.csv", parse_dates=["timestamp"])
    df = df.set_index("timestamp")
    df_smoothed = df.copy()
    for col in df.columns:
        df_smoothed[col] = validate_and_smooth(df[col], window=window_size, threshold=threshold_std)

    df_smoothed.to_csv(f"csv_by_baby_smooth/baby_{id}.csv")